# Deliverable 3 — Real Data: Ingestion & EDA
**Team 02 · Akshith Reddy Sheri (lead)** · Deep-dive: **time axis, validation scheme and date features**

> **NDA.** The AA files live only in the git-ignored `data/raw/` folder on my machine. This notebook prints
> **aggregates only, never individual records**. Before committing: **Kernel → Restart & Clear Output**.

**Sections**
1. Load
2. Shape and types
3. Missingness (values *and* missing route-months)
4. Target distribution and outliers
5. Leakage risk
6. **Time axis** — coverage, route completeness, trend and seasonality
7. **Validation scheme** — why the split must follow time
8. **Date features** — built by code, leakage-safe
9. Relationships and model-family view

## 1. Load

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

pd.set_option('display.max_columns', 40)
plt.rcParams['figure.figsize'] = (10, 4)

TARGET = 'OD_PAX'
RANDOM_STATE = 42
TEST_MONTHS = 12          # hold out the last 12 departure months for the time-based check

# Find the repo root (works whether Jupyter starts in the repo or in this folder)
repo_root = next((f for f in [Path.cwd(), *Path.cwd().parents]
                  if (f / 'DATA.md').is_file() and (f / 'data').is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('Open this notebook from inside the project repository.')

def find_file(name):
    for folder in [repo_root / 'data' / 'raw', repo_root / 'data']:
        if (folder / name).is_file():
            return folder / name
    raise FileNotFoundError(f'Put {name} in data/raw/ (git-ignored).')

pax = pd.read_csv(find_file('AA_PAX_FORECASTING.csv'))
try:
    airports = pd.read_csv(find_file('AA_AIRPORT_INFO.csv'))
except FileNotFoundError:
    airports = None
    print('Airport reference file not found -- airport checks will be skipped.')

required = {'DEP_YEAR', 'DEP_MONTH', 'ORG_AIRPRT', 'DES_AIRPRT', TARGET}
missing_cols = required - set(pax.columns)
if missing_cols:
    raise ValueError(f'Passenger file is missing expected columns: {sorted(missing_cols)}')
print('Files loaded locally. No individual records are displayed in this notebook.')

## 2. Shape and types

In [ ]:
for label, frame in [('Passenger', pax), ('Airport', airports)]:
    if frame is None:
        continue
    print(f'{label}: {frame.shape[0]:,} rows x {frame.shape[1]} columns')
    display(pd.DataFrame({'dtype': frame.dtypes.astype(str),
                          'distinct_values': frame.nunique(dropna=False)}))

index_like = [c for c in pax.columns
              if str(c).lower().startswith('unnamed') or str(c).lower() in ('index', 'id')]
print('Index-like columns (row numbers, not information):', index_like)

**Conclusion:** Each row is one origin–destination route in one departure month, with OD_PAX as the number of passengers. ORG_AIRPRT and DES_AIRPRT are airport codes (categories, not numbers), DEP_YEAR and DEP_MONTH are the departure period, and Unnamed: 0 is just a row number from the export, not real information. SF_LOAD_TMS is a system load timestamp. The airport file is a separate reference table keyed by airport code.

## 3. Missingness — values *and* missing route-months

In [ ]:
for label, frame in [('Passenger', pax), ('Airport', airports)]:
    if frame is None:
        continue
    summary = pd.DataFrame({'missing_count': frame.isna().sum(),
                            'missing_%': frame.isna().mean().mul(100).round(2)})
    print(label); display(summary[summary['missing_count'] > 0] if summary['missing_count'].any()
                          else 'no missing values')

if airports is not None and {'STATE_PROVNC_CD', 'CNTRY_CD'} <= set(airports.columns):
    by_country = (airports.assign(state_missing=airports['STATE_PROVNC_CD'].isna())
                  .groupby('CNTRY_CD', dropna=False)['state_missing'].agg(['size', 'mean']))
    print('Share of airports missing a state/province code, by country (top 10 by count):')
    display(by_country.sort_values('size', ascending=False).head(10).round(3))

In [ ]:
# Zero NaNs does not mean nothing is missing: a route with no row for a month is invisible to isna().
pax['period'] = pd.to_datetime(dict(year=pax['DEP_YEAR'], month=pax['DEP_MONTH'], day=1), errors='coerce')
pax['route'] = pax['ORG_AIRPRT'].astype(str) + '-' + pax['DES_AIRPRT'].astype(str)

all_periods = pd.period_range(pax['period'].min(), pax['period'].max(), freq='M')
months_per_route = pax.groupby('route')['period'].nunique()
print(f'{pax["route"].nunique():,} routes, {len(all_periods)} calendar months in range')
print(f'routes present in EVERY month: {(months_per_route == len(all_periods)).mean():.1%}')
print(f'route-months that would exist on a full calendar but have no row: '
      f'{1 - len(pax.drop_duplicates(["route", "period"])) / (pax["route"].nunique() * len(all_periods)):.1%}')
dupes = pax.duplicated(['route', 'period']).sum()
print(f'route-month combinations appearing more than once: {dupes:,}')

months_per_route.plot.hist(bins=30, title='How many months each route appears in')
plt.xlabel('months with a row'); plt.ylabel('routes'); plt.show()

**Conclusion:** The passenger table has no missing values, so no imputation is needed there. In the airport table, the missing state/province codes are not random: they belong to airports outside the US and Canada, so we fill them as "INTL" instead of dropping them. However, zero missing values does not mean nothing is missing: most routes appear in every month, so missing route-months are rare. The team should confirm with a PM whether a missing route-month means zero passengers or simply no record, because that changes how we build the training table.

## 4. Target distribution and outliers

In [ ]:
y = pd.to_numeric(pax[TARGET], errors='coerce')
display(y.describe().round(2).to_frame().T)
print(f'skew: {y.skew():.2f}   |   skew of log1p: {np.log1p(y.clip(lower=0)).skew():.2f}')
print(f'zero values: {(y == 0).mean():.2%}   |   negative values: {(y < 0).sum():,}')
q1, q3 = y.quantile([.25, .75]); iqr = q3 - q1
print(f'IQR outliers: {((y < q1 - 1.5 * iqr) | (y > q3 + 1.5 * iqr)).mean():.2%} of rows')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
y.plot.hist(bins=60, ax=axes[0], title=f'{TARGET}')
np.log1p(y.clip(lower=0)).plot.hist(bins=60, ax=axes[1], title=f'log1p({TARGET})')
plt.tight_layout(); plt.show()

**Conclusion:** OD_PAX is a passenger count with a strong right skew: most routes carry modest numbers and a few busy routes carry far more, while the log-transformed version is much more symmetric. The large values look like legitimate busy routes rather than errors, so we keep them. Because the target is a skewed count, a count-friendly loss such as Poisson or Tweedie is a better fit than plain squared error, and it avoids the underprediction Sahith raised on Aidan's PR when converting back from a log.

## 5. Leakage risk

Ask of every column: **would I have this value at the moment I need the forecast?**

In [ ]:
checks = []
if 'OD_REV_USD' in pax.columns:
    rev = pd.to_numeric(pax['OD_REV_USD'], errors='coerce')
    corr = rev.corr(y)
    implied_fare = (rev / y.replace(0, np.nan))
    checks.append({'column': 'OD_REV_USD',
                   'evidence': f'correlation with {TARGET}: {corr:.2f}; revenue / passengers is a fare-like ratio '
                               f'(median of the ratio is in the same units as a ticket price)',
                   'verdict': 'LEAK if it is realised revenue for the same route-month -- only known after travel'})
if 'SF_LOAD_TMS' in pax.columns:
    load_ts = pd.to_datetime(pax['SF_LOAD_TMS'], errors='coerce')
    after = (load_ts > pax['period'] + pd.offsets.MonthEnd(0)).mean()
    checks.append({'column': 'SF_LOAD_TMS',
                   'evidence': f'{after:.0%} of rows were loaded after their departure month ended',
                   'verdict': 'system load timestamp -- not information about demand; drop'})
for c in index_like:
    checks.append({'column': c, 'evidence': 'row number from the export',
                   'verdict': 'drop -- like the Deliverable 2 row counter'})
pd.DataFrame(checks)

**Conclusion:** I agree with Sahith and Aidan. OD_REV_USD is a leak: it moves almost in lockstep with OD_PAX and is realised revenue for the same route-month, which is only known after travel, so it must be dropped. SF_LOAD_TMS is a system load timestamp recorded after the departure month, not demand information, so we drop it. Unnamed: 0 is a row number and is dropped, like the Deliverable 2 row counter.

## 6. Time axis — my deep-dive

Kaggle had no dates. This data has `DEP_YEAR` and `DEP_MONTH`, so the questions are: how long is the history,
is there a trend, is there seasonality, and does the market change over time (routes opening/closing)?

In [ ]:
monthly = pax.groupby('period').agg(rows=('route', 'size'), routes=('route', 'nunique'),
                                    total_pax=(TARGET, 'sum'))
print(f'coverage: {monthly.index.min():%Y-%m} -> {monthly.index.max():%Y-%m} '
      f'({len(monthly)} months with data, {len(all_periods)} calendar months)')
gaps = sorted(set(all_periods.to_timestamp()) - set(monthly.index))
print('calendar months with NO data at all:', [f'{g:%Y-%m}' for g in gaps] or 'none')

fig, ax1 = plt.subplots()
monthly['total_pax'].plot(ax=ax1, marker='o', ms=3, label='total passengers')
ax1.set_ylabel('total passengers per month')
ax2 = ax1.twinx(); monthly['routes'].plot(ax=ax2, color='grey', ls='--', label='active routes')
ax2.set_ylabel('active routes')
ax1.set_title('Total passengers and active routes per departure month'); plt.show()

In [ ]:
# Seasonality: average month-of-year profile, scaled so each year is comparable
yearly_total = pax.groupby('DEP_YEAR')[TARGET].sum()
season = (pax.groupby(['DEP_YEAR', 'DEP_MONTH'])[TARGET].sum()
          .div(yearly_total, level='DEP_YEAR').unstack('DEP_YEAR'))
season.plot(marker='o', ms=3, title='Share of each year\'s passengers by month (one line per year)')
plt.xlabel('departure month'); plt.ylabel('share of year'); plt.xticks(range(1, 13)); plt.show()

full_years = pax.groupby('DEP_YEAR')['DEP_MONTH'].nunique().loc[lambda s: s == 12].index
yoy = yearly_total.loc[full_years].pct_change().mul(100).round(1)
print('Year-over-year change in total passengers (complete years only, %):')
print(yoy.dropna().to_string() if yoy.notna().any() else 'fewer than two complete years')

routes_by_period = pax.groupby('period')['route'].apply(set)
first, last = routes_by_period.iloc[0], routes_by_period.iloc[-1]
print(f'\nroutes in first month: {len(first):,} | last month: {len(last):,} | '
      f'new since first month: {len(last - first):,} | gone since first month: {len(first - last):,}')

**Conclusion — time axis**
- **Coverage:** The data covers about 7.5 years of departure months, with no empty months.
- **Trend:** Total passengers rise over time, with a sudden drop around 2020 and recovery after.
- **Seasonality:** The yearly lines follow the same shape every year, peaking in summer, so month of year carries real information.
- **Market changes:** The set of routes stays fairly stable, so the model mostly sees routes it already knows.

## 7. Validation scheme — why the split must follow time

Same model, same features, two ways of splitting. A random split lets the model see later months of the same
route while it is tested on earlier ones — information it would never have when forecasting. If the random split
scores much better, that gap is leakage from the future.

In [ ]:
base_cols = ['ORG_AIRPRT', 'DES_AIRPRT', 'DEP_YEAR', 'DEP_MONTH']
frame = pax.dropna(subset=['period']).sort_values('period')
X, y_all = frame[base_cols], pd.to_numeric(frame[TARGET], errors='coerce')
ok = y_all.notna(); X, y_all, periods = X[ok], y_all[ok], frame.loc[ok, 'period']

def make_model():
    prep = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
                               ['ORG_AIRPRT', 'DES_AIRPRT'])], remainder='passthrough')
    return Pipeline([('prep', prep), ('m', HistGradientBoostingRegressor(
        categorical_features=[0, 1], random_state=RANDOM_STATE))])

cutoff = periods.drop_duplicates().sort_values().iloc[-TEST_MONTHS]
train_t, test_t = periods < cutoff, periods >= cutoff
time_model = make_model().fit(X[train_t], y_all[train_t])
time_mae = mean_absolute_error(y_all[test_t], time_model.predict(X[test_t]))

X_tr, X_te, y_tr, y_te = train_test_split(X, y_all, test_size=test_t.mean(), random_state=RANDOM_STATE)
rand_mae = mean_absolute_error(y_te, make_model().fit(X_tr, y_tr).predict(X_te))
naive_mae = mean_absolute_error(y_all[test_t], np.full(test_t.sum(), y_all[train_t].median()))

result = pd.DataFrame({'MAE (passengers per row)': [rand_mae, time_mae, naive_mae]},
                      index=['random split (leaks the future)', f'time split (last {TEST_MONTHS} months held out)',
                             'naive: training median'])
display(result.round(1))
ratio = time_mae / rand_mae
if ratio > 1.1:
    print(f'the random split looks {ratio:.1f}x better than the honest time split -- that gap is leakage from the future')
else:
    print(f'the two splits score about the same here ({ratio:.2f}x) -- still use the time split: it is the only one '
          'that matches how the model will be used, and it protects later steps like target encoding')
print(f'held-out months start at {cutoff:%Y-%m}')

**Conclusion — validation scheme**
- The random split looked clearly better than the honest time split. That gap is how much a random split would have overstated our accuracy, because it lets the model see later months of the same routes.
- **Team rule:** train on earlier departure months and test on the most recent months. Fit every encoder, target encoding, and scaling step on the training months only, inside a Pipeline. For tuning in Deliverable 4, use rolling time folds (TimeSeriesSplit over months), never shuffled K-fold.

## 8. Date features — built by code, leakage-safe

Each feature must be **known at forecast time**. "Same month last year" for a route is known (it already
happened); anything from the forecast month itself is not.

In [ ]:
def build_date_features(frame):
    """Route-month passengers -> date features. Only uses the PAST, so it is safe to build before splitting."""
    g = (frame.groupby(['ORG_AIRPRT', 'DES_AIRPRT', 'period'], as_index=False)[TARGET].sum())
    g['DEP_YEAR'], g['DEP_MONTH'] = g['period'].dt.year, g['period'].dt.month
    g['month_sin'] = np.sin(2 * np.pi * g['DEP_MONTH'] / 12)
    g['month_cos'] = np.cos(2 * np.pi * g['DEP_MONTH'] / 12)
    g['months_since_start'] = ((g['period'].dt.year - g['period'].min().year) * 12
                               + g['period'].dt.month - g['period'].min().month)
    last_year = g[['ORG_AIRPRT', 'DES_AIRPRT', 'period', TARGET]].copy()
    last_year['period'] = last_year['period'] + pd.DateOffset(years=1)
    g = g.merge(last_year.rename(columns={TARGET: 'pax_same_month_last_year'}),
                on=['ORG_AIRPRT', 'DES_AIRPRT', 'period'], how='left')
    g['route_seen_last_year'] = g['pax_same_month_last_year'].notna().astype(int)
    return g

features = build_date_features(pax)
print('feature table:', features.shape)
print('columns:', features.columns.tolist())
print(f'rows with a same-month-last-year value: {features["route_seen_last_year"].mean():.1%} '
      '(the first year can never have one)')

# Does last year's value help? Re-run the honest time split with it added.
feat_cols = ['ORG_AIRPRT', 'DES_AIRPRT', 'DEP_YEAR', 'DEP_MONTH', 'pax_same_month_last_year']
train_f, test_f = features['period'] < cutoff, features['period'] >= cutoff
prep = ColumnTransformer([('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1),
                           ['ORG_AIRPRT', 'DES_AIRPRT'])], remainder='passthrough')
m = Pipeline([('prep', prep), ('m', HistGradientBoostingRegressor(categorical_features=[0, 1],
                                                                  random_state=RANDOM_STATE))])
m.fit(features.loc[train_f, feat_cols], features.loc[train_f, TARGET])
lag_mae = mean_absolute_error(features.loc[test_f, TARGET], m.predict(features.loc[test_f, feat_cols]))
print(f'time-split MAE without last-year feature: {time_mae:.1f} | with it: {lag_mae:.1f}')

**Date-feature hand-off** 
| Feature | Built from | Decision | Reason |
|---|---|---|---|
| `DEP_MONTH` | departure month | keep | captures seasonality (Section 6) |
| `month_sin`, `month_cos` | `DEP_MONTH` | optional | helps linear models treat December as next to January; trees do not need it |
| `DEP_YEAR` | departure year | keep, with care | captures the trend, but trees cannot extrapolate to a year they never saw |
| `months_since_start` | year + month | optional | a smooth time index for the trend |
| `pax_same_month_last_year` | the route's own history | keep: it lowered the time-split error | known at forecast time; carries route size and seasonality |
| `route_seen_last_year` | the same | keep | flags new routes with no history |
| `SF_LOAD_TMS` | load timestamp | drop | system metadata, not demand information |

## 9. Relationships and model-family view

In [ ]:
print(f'origin airports: {pax["ORG_AIRPRT"].nunique():,} | destinations: {pax["DES_AIRPRT"].nunique():,} | '
      f'routes: {pax["route"].nunique():,}')
num = features[['DEP_YEAR', 'DEP_MONTH', 'months_since_start', 'pax_same_month_last_year', TARGET]]
display(pd.DataFrame({'pearson': num.corr()[TARGET], 'spearman': num.corr(method='spearman')[TARGET]})
        .drop(TARGET).round(3))

**My model-family view**

Gradient-boosted trees (LightGBM or XGBoost) with a count-friendly objective (Poisson or Tweedie), validated on a time split:
1. **High-cardinality categories:** there are many origin and destination airports and even more routes. Trees with native categorical handling deal with this well, while one-hot encoding for a linear model would create a huge, sparse table (Section 9).
2. **Non-linear time patterns:** demand peaks in specific months and differs by route, which trees capture through interactions without hand-built features (Section 6).
3. **Skewed count target:** OD_PAX is a right-skewed count (Section 4), and boosted trees support Poisson and Tweedie losses directly.

**What could change this:** if the "same month last year" feature dominates, a simpler model using it may perform almost as well, so we should compare against that baseline in Deliverable 4.

---
### Before committing
1. **Kernel → Restart & Clear Output**, then save
2. `git status` must show only this notebook — nothing from `data/`